In [1]:
from pyspark.sql import SparkSession
import spark

In [3]:
spark = SparkSession.builder \
        .appName("Managed VS External Table")\
        .enableHiveSupport() \
        .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/13 16:32:14 INFO SparkEnv: Registering MapOutputTracker
26/04/13 16:32:14 INFO SparkEnv: Registering BlockManagerMaster
26/04/13 16:32:14 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/13 16:32:14 INFO SparkEnv: Registering OutputCommitCoordinator


In [4]:
df = spark.read \
    .format('csv') \
    .option('inferSchema', 'true') \
    .option('header', 'true') \
    .load('/data/customers_150mb.csv')


In [5]:
df.show(5)

+-----------+----------+---------+---------+-------+-----------------+---------+
|customer_id|      name|     city|    state|country|registration_date|is_active|
+-----------+----------+---------+---------+-------+-----------------+---------+
|          0|Customer_0|  Kolkata|Karnataka|  India|       2023-12-21|    false|
|          1|Customer_1|Ahmedabad|Telangana|  India|       2023-03-07|     true|
|          2|Customer_2|Bangalore|Karnataka|  India|       2023-01-14|    false|
|          3|Customer_3|  Chennai|Karnataka|  India|       2023-11-16|    false|
|          4|Customer_4|    Delhi|Telangana|  India|       2023-05-23|     true|
+-----------+----------+---------+---------+-------+-----------------+---------+
only showing top 5 rows



#### Create a temporary view on the above table

#### Managed Table

In [6]:
df.createOrReplaceTempView('temp_customers')

In [8]:
spark.sql('select * from temp_customers limit 5').show()

+-----------+----------+---------+---------+-------+-----------------+---------+
|customer_id|      name|     city|    state|country|registration_date|is_active|
+-----------+----------+---------+---------+-------+-----------------+---------+
|          0|Customer_0|  Kolkata|Karnataka|  India|       2023-12-21|    false|
|          1|Customer_1|Ahmedabad|Telangana|  India|       2023-03-07|     true|
|          2|Customer_2|Bangalore|Karnataka|  India|       2023-01-14|    false|
|          3|Customer_3|  Chennai|Karnataka|  India|       2023-11-16|    false|
|          4|Customer_4|    Delhi|Telangana|  India|       2023-05-23|     true|
+-----------+----------+---------+---------+-------+-----------------+---------+



In [9]:
spark.sql('show tables').show()

ivysettings.xml file not found in HIVE_HOME or HIVE_CONF_DIR,/etc/hive/conf.dist/ivysettings.xml will be used


+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|  default|           customers|      false|
|  default|customers_persistent|      false|
|         |      temp_customers|       true|
+---------+--------------------+-----------+



In [10]:
spark.sql('''create table managed_customers as 
                select * from temp_customers''')

26/04/13 16:38:49 WARN ResolveSessionCatalog: A Hive serde table will be created as there is no table provider specified. You can set spark.sql.legacy.createHiveTableByDefault to false so that native data source table will be created instead.
26/04/13 16:38:49 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


DataFrame[]

In [16]:
spark.sql('describe extended managed_customers').show(truncate = False)

+----------------------------+----------------------------------+-------+
|col_name                    |data_type                         |comment|
+----------------------------+----------------------------------+-------+
|customer_id                 |int                               |NULL   |
|name                        |string                            |NULL   |
|city                        |string                            |NULL   |
|state                       |string                            |NULL   |
|country                     |string                            |NULL   |
|registration_date           |date                              |NULL   |
|is_active                   |boolean                           |NULL   |
|                            |                                  |       |
|# Detailed Table Information|                                  |       |
|Catalog                     |spark_catalog                     |       |
|Database                    |default 

In [17]:
!hadoop fs -ls /user/hive/warehouse/

Found 4 items
drwxr-xr-x   - root hadoop          0 2026-04-13 16:14 /user/hive/warehouse/customers
drwxr-xr-x   - root hadoop          0 2026-04-13 15:05 /user/hive/warehouse/customers_persistent
drwxr-xr-x   - root hadoop          0 2026-04-13 16:05 /user/hive/warehouse/ecommerce.db
drwxr-xr-x   - root hadoop          0 2026-04-13 16:39 /user/hive/warehouse/managed_customers


#### External Table

In [18]:
!hadoop fs -ls /data/

Found 7 items
-rw-r--r--   2 root hadoop    1060750 2026-04-10 21:42 /data/customers.csv
-rw-r--r--   2 root hadoop       5488 2026-04-10 15:25 /data/customers_100.csv
-rw-r--r--   2 root hadoop   10528211 2026-04-10 21:41 /data/customers_10mb.csv
-rw-r--r--   2 root hadoop  168541068 2026-04-10 21:40 /data/customers_150mb.csv
-rw-r--r--   2 root hadoop  236978176 2026-04-10 16:31 /data/customers_300mb.csv
-rw-r--r--   2 root hadoop        209 2026-04-11 17:27 /data/dates_data.csv
drwxr-xr-x   - root hadoop          0 2026-04-11 15:48 /data/write_output.csv


In [19]:
spark.sql('''
create external table external_customers 
using csv
location '/data/customers.csv/'
''')


26/04/13 16:45:11 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider csv. Persisting data source table `spark_catalog`.`default`.`external_customers` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.


DataFrame[]

In [20]:
spark.sql('describe extended external_customers').show(truncate=False)

+----------------------------+--------------------------------------------------+-------+
|col_name                    |data_type                                         |comment|
+----------------------------+--------------------------------------------------+-------+
|_c0                         |string                                            |NULL   |
|_c1                         |string                                            |NULL   |
|_c2                         |string                                            |NULL   |
|_c3                         |string                                            |NULL   |
|_c4                         |string                                            |NULL   |
|_c5                         |string                                            |NULL   |
|_c6                         |string                                            |NULL   |
|                            |                                                  |       |
|# Detaile

In [ ]:
spark.sql('''
create external table external_customers_2 (
customer_id INT,
name STRING,
city STRING,
state string,
country string,
registration_date STRING,
is_active BOOLEAN
)
using csv
location '/data/external_data/'
''')


In [21]:
spark.stop()